# Import thư viện, đọc và merge file

In [ ]:
import pandas as pd
import numpy as np

path_phase_files = [
    "/kaggle/input/ds317/phase_1_sentiment.csv",
    "/kaggle/input/ds317/phase_2_sentiment.csv",
    "/kaggle/input/ds317/phase_3 _sentiment.csv",
    "/kaggle/input/ds317/phase_4 _sentiment.csv"
]
phase_dfs = [pd.read_csv(path) for path in path_phase_files]
df_comment = pd.concat(phase_dfs, ignore_index=True)

print()

file_path_enroll = "/kaggle/input/mooccubex-cleaned/Label_3_and_5_classes(have_course_info).csv"
df_enroll = pd.read_csv(file_path_enroll)

In [ ]:
df_comment.info()

# Merge file

In [ ]:
# Merge df_reply với df_enroll
merged_df = df_enroll.merge(
    df_comment,
    on=["user_id", "course_id", "enroll_time"],
    how="inner"
)
merged_df = merged_df.loc[:, ~merged_df.columns.str.startswith('Unnamed:')]
print("Trước khi merge:", len(df_enroll))
print("Sau khi merge:", len(merged_df))

# Lưu merged_df ra file csv
merged_file = "/kaggle/working/merged_df.csv"
merged_df.to_csv(merged_file, index=False)
print(f"Đã lưu merged_df ra file: {merged_file}")

display(merged_df.head())

In [ ]:
merged_df.info()

# Chia dữ liệu thành 4 phase

## Hàm gán phase

In [ ]:
df = merged_df.copy()
print(f"Số dòng ban đầu: {len(df)}")
# Hàm gán phase
def assign_phase(row):
    days = row['days_since_enroll']
    course_duration = row['enrollment_to_end']
    if days <= 0.25 * course_duration:
        return 'Phase_1'
    elif days <= 0.50 * course_duration:
        return 'Phase_2'
    elif days <= 0.75 * course_duration:
        return 'Phase_3'
    elif days <= 0.9 * course_duration:
        return 'Phase_4'
    else:
        return None

## Thực hiện gán phase

In [ ]:
# Đọc file
print("Đang đọc merged_df.csv...")
df = merged_df.copy()
print(f"Số dòng ban đầu: {len(df)}")
# Gán giá trị phase cho từng dòng dựa vào số ngày kể từ khi ghi danh
print("Đang gán phase...")
df['phase'] =  df.apply(assign_phase, axis=1)  # Hàm assign_phase sẽ phân loại phase (1, 2, 3, 4)

df = df[df['phase'].notna()]
print(f"Số dòng sau khi gán phase: {len(df)}")

print("\nPhân bố phase:")
print(df['phase'].value_counts().sort_index())  # Đếm số dòng theo từng phase

## Kiểm tra trùng và groupby

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import entropy

# Giả sử df đã có sẵn trong bộ nhớ
phases = ['Phase_1', 'Phase_2', 'Phase_3', 'Phase_4']
group_cols = ['user_id', 'course_id', 'enroll_time']
agg_results = {}

for i, phase in enumerate(phases, start=1):
    print(f"\n=== {phase} ===")
    df_phase = df[df['phase'] == phase].copy()
    
    # Bỏ cột không cần
    drop_cols = ['days_since_enroll', 'resource_id', 'id', 'comment_id']
    df_phase = df_phase.drop(columns=drop_cols, errors='ignore')
    
    # Kiểm tra trùng
    num_dups_before = df_phase.duplicated(subset=group_cols, keep=False).sum()
    print(f"Trước khi gộp: {num_dups_before} bản ghi trùng dựa trên {group_cols}.")
    
    # Định nghĩa sentiment columns
    sentiment_cols = ['negative', 'neutral', 'positive']
    
    # Đếm sentiment (tổng số bình luận mang cảm xúc)
    sentiment_counts = (
        df_phase
        .groupby(group_cols, as_index=False)[sentiment_cols]
        .sum()
        .rename(columns={c: f"total_{c}" for c in sentiment_cols})
    )

    # ==== 1. Tính các đặc trưng thống kê tổng hợp ====
    numeric_cols = df_phase.select_dtypes(include=['int64', 'float64']).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in sentiment_cols]

    sum_cols = ['num_words', 'comment_count', 'total_words']
    mean_cols = [c for c in numeric_cols if c not in sum_cols]
    
    agg_dict = {col: 'mean' for col in mean_cols}
    for c in sum_cols:
        if c in numeric_cols:  # Kiểm tra cột tồn tại
            agg_dict[c] = 'sum'
    
    # Gộp numeric cơ bản
    agg_numeric = (
        df_phase
        .groupby(group_cols, as_index=False)
        .agg(agg_dict)
        .sort_values(by=group_cols)
        .reset_index(drop=True)
    )
    
    # ==== 2. Các đặc trưng ngôn ngữ & thời gian ====
    def extract_comment_features(g):
        # Tính text_diversity đúng hơn (nếu có cột text)
        # Hoặc dùng tỷ lệ unique words nếu có
        total_words = g['total_words'].sum() if 'total_words' in g.columns else 0
        num_comments = len(g)
        
        # Xử lý mode an toàn hơn
        time_bin_mode = g['time_bin'].mode()
        most_common_time = time_bin_mode.iloc[0] if len(time_bin_mode) > 0 else np.nan
        
        return pd.Series({
            'total_comments': num_comments,
            'avg_comment_length': g['num_words'].mean() if 'num_words' in g.columns else 0,
            'std_comment_length': g['num_words'].std() if 'num_words' in g.columns else 0,
            'max_comment_length': g['num_words'].max() if 'num_words' in g.columns else 0,
            'text_diversity': total_words / num_comments if num_comments > 0 else 0,
            'comment_days_active': g['create_time'].astype(str).str.split(" ").str[0].nunique() if 'create_time' in g.columns else 0,
            'entropy_time_mean': g['entropy_time_comment'].mean() if 'entropy_time_comment' in g.columns else 0,
            'most_common_time_bin': most_common_time,
            'time_entropy_var': g['entropy_time_comment'].var() if 'entropy_time_comment' in g.columns else 0,
        })

    comment_features = (
        df_phase
        .groupby(group_cols, group_keys=False)
        .apply(extract_comment_features, include_groups=False)  # ✅ Thêm include_groups=False
        .reset_index()
    )

    # ==== 3. Tỷ lệ cảm xúc và entropy ====
    def compute_sentiment_ratios(row):
        # Xử lý NaN trước khi tính
        neg = row.get('total_negative', 0) or 0
        neu = row.get('total_neutral', 0) or 0
        pos = row.get('total_positive', 0) or 0
        
        total = neg + neu + pos
        if total == 0:
            return pd.Series({
                'positive_ratio': 0.0,
                'neutral_ratio': 0.0,
                'negative_ratio': 0.0,
                'sentiment_entropy': 0.0
            })
        
        p = np.array([pos / total, neu / total, neg / total])
        # Loại bỏ giá trị 0 để tránh log(0) trong entropy
        p_nonzero = p[p > 0]
        
        return pd.Series({
            'positive_ratio': float(p[0]),
            'neutral_ratio': float(p[1]),
            'negative_ratio': float(p[2]),
            'sentiment_entropy': float(entropy(p_nonzero)) if len(p_nonzero) > 0 else 0.0
        })

    # Gộp 3 phần
    agg_df = (
        agg_numeric
        .merge(sentiment_counts, on=group_cols, how='left')
        .merge(comment_features, on=group_cols, how='left')
    )

    # Fill NaN cho sentiment counts trước khi tính tỷ lệ
    sentiment_total_cols = ['total_negative', 'total_neutral', 'total_positive']
    for col in sentiment_total_cols:
        if col in agg_df.columns:
            agg_df[col] = agg_df[col].fillna(0)

    # Tính thêm tỷ lệ cảm xúc
    sentiment_ratios = agg_df.apply(compute_sentiment_ratios, axis=1)
    agg_df = pd.concat([agg_df, sentiment_ratios], axis=1)

    # Bổ sung phase_num (tránh ghi đè cột phase gốc)
    agg_df['phase_num'] = i

    # ==== Xử lý giá trị thiếu (fillna) ====
    # Các cột số: thay NaN bằng 0
    num_cols = agg_df.select_dtypes(include=['int64', 'float64']).columns
    agg_df[num_cols] = agg_df[num_cols].fillna(0)

    # Các cột object (chuỗi): thay NaN bằng "unknown"
    obj_cols = agg_df.select_dtypes(include=['object']).columns
    agg_df[obj_cols] = agg_df[obj_cols].fillna("unknown")

    # Drop các cột không cần thiết dạng Unnamed
    agg_df = agg_df.loc[:, ~agg_df.columns.str.startswith('Unnamed:')]

    # Lưu kết quả từng phase
    agg_results[i] = agg_df
    
    print(f"✓ Phase {i}: {len(agg_df)} records after aggregation")

print("\n✅ Hoàn tất gộp dữ liệu, tạo đặc trưng và xử lý NaN cho tất cả 4 phase.")

In [ ]:
# Danh sách các cột cần loại bỏ
drop_cols = [
    'enroll_time', 'video', 'assignment', 'exam',
    'amount_exercise_each_exam', 'total_correct_ratio_exam',
    'average_correct_ratio_exam', 'amount_exercise_each_assignment',
    'total_correct_ratio_assignment', 'average_correct_ratio_assignment',
    'watched_videos', 'video_counts', 'watch_percent', 'score',
    'course_duration', 'enrollment_to_end', 'comment_count','num_words','phase_num'
]

# Loại bỏ cột trong từng DataFrame của agg_results
for key, df_agg in agg_results.items():
    agg_results[key] = df_agg.drop(columns=drop_cols, errors='ignore')

print("✅ Đã loại bỏ các cột không cần thiết khỏi toàn bộ DataFrame trong agg_results.")


In [ ]:
agg_results_renamed=[]
for i in range(1,len(agg_results)+1):
    df = agg_results[i]
    
    # Chỉ xử lý nếu phần tử là DataFrame
    if isinstance(df, pd.DataFrame):
        df_new = df.copy()
        
        # Tạo mapping: thêm hậu tố _phaseN (loại trừ course_id, user_id)
        new_columns = {
            col: f"{col}_phase{i}" for col in df_new.columns
            if col not in ['course_id', 'user_id']
        }
        
        # Đổi tên cột
        df_new = df_new.rename(columns=new_columns)
        
        # Lưu lại
        agg_results_renamed.append(df_new)
    else:
        print(f"⚠️ Bỏ qua phần tử thứ {i} (kiểu {type(df).__name__}) vì không phải DataFrame.")

In [ ]:
agg_results_renamed[1].columns

In [ ]:
agg_results_renamed[0].columns

## Chia file

In [ ]:
# Lưu từng phase vào file CSV
for i, df in enumerate(agg_results_renamed):
    filename = f"comment_phase_{i+1}.csv"  # phase bắt đầu từ 1
    df.to_csv(filename, index=False)
    print(f"Đã lưu {filename}: {len(df)} dòng, {len(df.columns)} cột")

print("\n✅ Hoàn tất xử lý và lưu tất cả các phase.")


# Kiểm tra lại file

In [1]:
path_phase_1 = "/kaggle/working/comment_phase_1.csv"
path_phase_2 = "/kaggle/working/comment_phase_2.csv"
path_phase_3 = "/kaggle/working/comment_phase_3.csv"
path_phase_4 = "/kaggle/working/comment_phase_4.csv"

## Phase 1

In [4]:
df_phase_1 = pd.read_csv(path_phase_1)

print(df_phase_1.info())
df_phase_1.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219232 entries, 0 to 219231
Data columns (total 21 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   user_id                      219232 non-null  object 
 1   course_id                    219232 non-null  object 
 2   time_bin_phase1              219232 non-null  float64
 3   entropy_time_comment_phase1  219232 non-null  float64
 4   total_words_phase1           219232 non-null  float64
 5   total_negative_phase1        219232 non-null  int64  
 6   total_neutral_phase1         219232 non-null  int64  
 7   total_positive_phase1        219232 non-null  int64  
 8   total_comments_phase1        219232 non-null  float64
 9   avg_comment_length_phase1    219232 non-null  float64
 10  std_comment_length_phase1    219232 non-null  float64
 11  max_comment_length_phase1    219232 non-null  float64
 12  text_diversity_phase1        219232 non-null  float64
 13 

,user_id,course_id,time_bin_phase1,entropy_time_comment_phase1,total_words_phase1,total_negative_phase1,total_neutral_phase1,total_positive_phase1,total_comments_phase1,avg_comment_length_phase1,...,max_comment_length_phase1,text_diversity_phase1,comment_days_active_phase1,entropy_time_mean_phase1,most_common_time_bin_phase1,time_entropy_var_phase1,positive_ratio_phase1,neutral_ratio_phase1,negative_ratio_phase1,sentiment_entropy_phase1
0,U_1001314,C_1779421,43.333333,0.636514,129.0,0,2,1,3.0,14.333333,...,22.0,43.0,2.0,0.636514,43.0,0.0,0.333333,0.666667,0.0,0.636514
1,U_1001625,C_735164,19.000000,0.000000,8.0,1,0,0,1.0,8.000000,...,8.0,8.0,1.0,0.000000,19.0,0.0,0.000000,0.000000,1.0,0.000000
2,U_1001703,C_735164,34.000000,0.693147,4.0,0,0,2,2.0,1.000000,...,1.0,2.0,2.0,0.693147,23.0,0.0,1.000000,0.000000,0.0,0.000000
3,U_1001806,C_1903976,31.000000,0.000000,5.0,0,1,0,1.0,5.000000,...,5.0,5.0,1.0,0.000000,31.0,0.0,0.000000,1.000000,0.0,0.000000
4,U_1002046,C_735164,27.000000,0.000000,47.0,0,0,1,1.0,47.000000,...,47.0,47.0,1.0,0.000000,27.0,0.0,1.000000,0.000000,0.0,0.000000


In [5]:
df_phase_1.describe()

,time_bin_phase1,entropy_time_comment_phase1,total_words_phase1,total_negative_phase1,total_neutral_phase1,total_positive_phase1,total_comments_phase1,avg_comment_length_phase1,std_comment_length_phase1,max_comment_length_phase1,text_diversity_phase1,comment_days_active_phase1,entropy_time_mean_phase1,most_common_time_bin_phase1,time_entropy_var_phase1,positive_ratio_phase1,neutral_ratio_phase1,negative_ratio_phase1,sentiment_entropy_phase1
count,219232.000000,219232.000000,2.192320e+05,219232.000000,219232.000000,219232.000000,219232.000000,219232.000000,219232.000000,219232.000000,219232.000000,219232.000000,219232.000000,219232.000000,2.192320e+05,219232.000000,219232.000000,219232.000000,219232.000000
mean,29.902263,0.734307,2.261382e+03,0.235157,5.337004,2.039479,7.611640,15.149136,9.147092,31.614769,93.388232,2.026611,0.734307,28.857566,2.759735e-02,0.274252,0.685423,0.040325,0.343611
std,7.953275,0.718678,1.542242e+05,0.662448,11.712203,9.458183,16.302443,28.427195,21.729164,62.498780,270.242450,1.461237,0.718678,9.551809,9.850058e-02,0.318478,0.332100,0.134847,0.341474
min,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
25%,24.357143,0.000000,1.900000e+01,0.000000,1.000000,0.000000,2.000000,3.076923,0.000000,5.000000,10.000000,1.000000,0.000000,22.000000,0.000000e+00,0.000000,0.500000,0.000000,0.000000
50%,30.000000,0.659167,1.950000e+02,0.000000,3.000000,1.000000,5.000000,6.000000,2.507023,12.000000,37.000000,2.000000,0.659167,29.000000,0.000000e+00,0.166667,0.766667,0.000000,0.348832
75%,35.166667,1.226556,7.840000e+02,0.000000,7.000000,3.000000,11.000000,13.125000,7.622478,28.000000,83.673077,3.000000,1.226556,36.000000,5.546678e-32,0.466667,1.000000,0.000000,0.655482
max,47.000000,3.507799,4.504874e+07,116.000000,3775.000000,2361.000000,4059.000000,820.000000,653.366666,1929.000000,23774.000000,29.000000,3.507799,47.000000,3.086039e+00,1.000000,1.000000,1.000000,1.098612


## Phase 2

In [6]:
df_phase_2 = pd.read_csv(path_phase_2)

print(df_phase_2.info())
df_phase_2.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97195 entries, 0 to 97194
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   user_id                      97195 non-null  object 
 1   course_id                    97195 non-null  object 
 2   time_bin_phase2              97195 non-null  float64
 3   entropy_time_comment_phase2  97195 non-null  float64
 4   total_words_phase2           97195 non-null  float64
 5   total_negative_phase2        97195 non-null  int64  
 6   total_neutral_phase2         97195 non-null  int64  
 7   total_positive_phase2        97195 non-null  int64  
 8   total_comments_phase2        97195 non-null  float64
 9   avg_comment_length_phase2    97195 non-null  float64
 10  std_comment_length_phase2    97195 non-null  float64
 11  max_comment_length_phase2    97195 non-null  float64
 12  text_diversity_phase2        97195 non-null  float64
 13  comment_days_act

,user_id,course_id,time_bin_phase2,entropy_time_comment_phase2,total_words_phase2,total_negative_phase2,total_neutral_phase2,total_positive_phase2,total_comments_phase2,avg_comment_length_phase2,...,max_comment_length_phase2,text_diversity_phase2,comment_days_active_phase2,entropy_time_mean_phase2,most_common_time_bin_phase2,time_entropy_var_phase2,positive_ratio_phase2,neutral_ratio_phase2,negative_ratio_phase2,sentiment_entropy_phase2
0,U_1001314,C_1779421,43.000000,0.000000,384.0,0,2,0,2.0,96.000000,...,151.0,192.0,1.0,0.000000,43.0,0.000000,0.000000,1.000000,0.000000,0.000000
1,U_1001703,C_735164,17.000000,0.000000,78.0,0,1,1,2.0,19.500000,...,28.0,39.0,1.0,0.000000,17.0,0.000000,0.500000,0.500000,0.000000,0.693147
2,U_10030628,C_936971,32.222222,1.149060,360.0,0,9,0,9.0,4.444444,...,16.0,40.0,4.0,1.149060,35.0,0.000000,0.000000,1.000000,0.000000,0.000000
3,U_10031042,C_936971,44.333333,0.636514,105.0,1,0,2,3.0,11.666667,...,20.0,35.0,2.0,0.636514,44.0,0.000000,0.666667,0.000000,0.333333,0.636514
4,U_10031254,C_936971,40.142857,0.792168,189.0,0,4,3,7.0,8.428571,...,24.0,27.0,3.0,0.792168,40.0,0.082169,0.428571,0.571429,0.000000,0.682908


## Phase 3

In [7]:
df_phase_3 = pd.read_csv(path_phase_3)

print(df_phase_3.info())
df_phase_3.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41402 entries, 0 to 41401
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   user_id                      41402 non-null  object 
 1   course_id                    41402 non-null  object 
 2   time_bin_phase3              41402 non-null  float64
 3   entropy_time_comment_phase3  41402 non-null  float64
 4   total_words_phase3           41402 non-null  float64
 5   total_negative_phase3        41402 non-null  int64  
 6   total_neutral_phase3         41402 non-null  int64  
 7   total_positive_phase3        41402 non-null  int64  
 8   total_comments_phase3        41402 non-null  float64
 9   avg_comment_length_phase3    41402 non-null  float64
 10  std_comment_length_phase3    41402 non-null  float64
 11  max_comment_length_phase3    41402 non-null  float64
 12  text_diversity_phase3        41402 non-null  float64
 13  comment_days_act

,user_id,course_id,time_bin_phase3,entropy_time_comment_phase3,total_words_phase3,total_negative_phase3,total_neutral_phase3,total_positive_phase3,total_comments_phase3,avg_comment_length_phase3,...,max_comment_length_phase3,text_diversity_phase3,comment_days_active_phase3,entropy_time_mean_phase3,most_common_time_bin_phase3,time_entropy_var_phase3,positive_ratio_phase3,neutral_ratio_phase3,negative_ratio_phase3,sentiment_entropy_phase3
0,U_10034409,C_936971,47.000000,0.000000,190.0,0,7,3,10.0,1.900000,...,4.0,19.0,1.0,0.000000,47.0,0.0,0.300000,0.700000,0.0,0.610864
1,U_10034455,C_936971,38.000000,0.000000,364.0,0,5,2,7.0,7.428571,...,14.0,52.0,1.0,0.000000,38.0,0.0,0.285714,0.714286,0.0,0.598270
2,U_10034661,C_936971,41.000000,0.000000,4.0,0,1,0,1.0,4.000000,...,4.0,4.0,1.0,0.000000,41.0,0.0,0.000000,1.000000,0.0,0.000000
3,U_1005189,C_735164,22.333333,0.636514,165.0,0,1,2,3.0,18.333333,...,38.0,55.0,1.0,0.636514,22.0,0.0,0.666667,0.333333,0.0,0.636514
4,U_1005398,C_735164,38.000000,0.000000,416.0,0,0,4,4.0,26.000000,...,45.0,104.0,1.0,0.000000,38.0,0.0,1.000000,0.000000,0.0,0.000000


## Phase 4

In [8]:
df_phase_4 = pd.read_csv(path_phase_4)

print(df_phase_4.info())
df_phase_4.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11122 entries, 0 to 11121
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   user_id                      11122 non-null  object 
 1   course_id                    11122 non-null  object 
 2   time_bin_phase4              11122 non-null  float64
 3   entropy_time_comment_phase4  11122 non-null  float64
 4   total_words_phase4           11122 non-null  float64
 5   total_negative_phase4        11122 non-null  int64  
 6   total_neutral_phase4         11122 non-null  int64  
 7   total_positive_phase4        11122 non-null  int64  
 8   total_comments_phase4        11122 non-null  float64
 9   avg_comment_length_phase4    11122 non-null  float64
 10  std_comment_length_phase4    11122 non-null  float64
 11  max_comment_length_phase4    11122 non-null  float64
 12  text_diversity_phase4        11122 non-null  float64
 13  comment_days_act

,user_id,course_id,time_bin_phase4,entropy_time_comment_phase4,total_words_phase4,total_negative_phase4,total_neutral_phase4,total_positive_phase4,total_comments_phase4,avg_comment_length_phase4,...,max_comment_length_phase4,text_diversity_phase4,comment_days_active_phase4,entropy_time_mean_phase4,most_common_time_bin_phase4,time_entropy_var_phase4,positive_ratio_phase4,neutral_ratio_phase4,negative_ratio_phase4,sentiment_entropy_phase4
0,U_1002046,C_735164,33.0,0.000000,168.0,0,0,3,3.0,18.666667,...,19.0,56.0,1.0,0.000000,33.0,0.0,1.0,0.0,0.0,0.000000
1,U_1004686,C_735164,32.0,0.000000,76.0,0,0,2,2.0,19.000000,...,25.0,38.0,1.0,0.000000,32.0,0.0,1.0,0.0,0.0,0.000000
2,U_1004755,C_735164,35.0,1.609438,228.0,0,0,4,4.0,10.500000,...,16.0,57.0,3.0,1.609438,22.0,0.0,1.0,0.0,0.0,0.000000
3,U_10069480,C_707456,24.0,0.636514,56.0,0,1,1,2.0,10.000000,...,13.0,28.0,1.0,0.636514,24.0,0.0,0.5,0.5,0.0,0.693147
4,U_1008115,C_735164,41.0,0.000000,63.0,0,0,3,3.0,7.000000,...,7.0,21.0,1.0,0.000000,41.0,0.0,1.0,0.0,0.0,0.000000
